In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import os.path as path
import utils
import glob

In [2]:
nuts_df = gpd.read_file(path.join(utils.raw_data_dir, "NUTS_RG_01M_2021_4326.shp"))

In [3]:
nuts_df = nuts_df.to_crs(epsg=3035)
nuts_df["area_km2"] = round(nuts_df.geometry.area / 1e6, 2)

In [4]:
population_data = pd.read_csv(path.join(utils.raw_data_dir, "estat_demo_r_pjangrp3.tsv"))
population_data = population_data[population_data["sex"] == "T"]
population_data = population_data[population_data["age"] == "TOTAL"]

In [5]:
last_col = population_data.columns[-1]

population_data[last_col] = population_data[last_col].apply(lambda x: [e for e in x.split("\t") if e != ": " ])
population_data["NUTS_ID"] = population_data[last_col].apply(lambda x: x[0])
population_data["population"] = population_data[last_col].apply(lambda x: x[-1])
population_data = population_data.drop(columns=[last_col, "sex", "age", "freq", "unit"])

In [6]:
nuts_df = pd.merge(nuts_df, population_data, on="NUTS_ID", how="outer")

In [7]:
nuts3_df = nuts_df[nuts_df["LEVL_CODE"] == 3].drop(columns=["LEVL_CODE", "MOUNT_TYPE", "URBN_TYPE", "COAST_TYPE"])

In [8]:
crop_profile_files = glob.glob(path.join(utils.intermediate_data_dir, "nuts3_crop_profile", "*.geojson"))
crop_profile_df = pd.concat([gpd.read_file(f) for f in crop_profile_files])

In [9]:
def calc_crop_area(in_dict):
    in_dict = eval(in_dict)
    out_dict = {}
    for k, v in in_dict.items():
        if k == 0:
            # default value for non-cropland pixels is 0
            continue
        crop_name = utils.cropland_type_dict[k]
        # NOTE: this next step is important and deserves explanation
        # the raster size of the cropland dataset is _precisely_ 10x10m for all pixels, as defined by the CRS
        # thus, each pixel has an area of 100m^2. We get the total area in m^2 by multiplying the pixel value by 100
        # to get from m^2->km^2 we need to divide by 1.000*1.000, i.e. 1.000.000
        # in other words, we divide by 10.000 or 1e4
        area_km = round(v * 1e-4, 2)
        out_dict[crop_name] = area_km
    return out_dict

crop_profile_df["cropland_km2_by_type"] = crop_profile_df["crop_profile"].apply(calc_crop_area)
crop_profile_df["cropland_km2"] = crop_profile_df["cropland_km2_by_type"].apply(lambda x: round(sum(x.values()), 2))


In [10]:
crop_profile_df = crop_profile_df[["NUTS_ID", "cropland_km2", "cropland_km2_by_type"]]
nuts3_df = pd.merge(nuts3_df, crop_profile_df, on="NUTS_ID", how="outer")
nuts3_df["cropland_area_percent"] = round(100 * nuts3_df["cropland_km2"] / nuts3_df["area_km2"], 2)

In [11]:
nuts3_drought_data = gpd.read_file(
    path.join(utils.out_data_dir, "drought_days_nuts3.geojson")
)
nuts3_drought_data = nuts3_drought_data[
    [
        "NUTS_ID",
        "median_warning_days",
        "max_warning_days",
        "max_warning_days_year",
        "median_alert_days",
        "max_alert_days",
        "max_alert_days_year",
        "median_drought_days",
        "max_drought_days",
        "max_drought_days_year",
    ]
]

In [12]:
nuts3_df = gpd.pd.merge(nuts3_df, nuts3_drought_data, on="NUTS_ID", how="outer")

In [13]:
nuts3_df = nuts3_df[
    [
        "NUTS_ID",
        "CNTR_CODE",
        "NUTS_NAME",
        "population",
        "area_km2",
        "cropland_km2",
        "cropland_area_percent",
        "cropland_km2_by_type",
        "NAME_LATN",
        "median_drought_days",
        "max_drought_days",
        "max_drought_days_year",
        "median_warning_days",
        "max_warning_days",
        "max_warning_days_year",
        "median_alert_days",
        "max_alert_days",
        "max_alert_days_year",
        "geometry",
    ]
]
nuts3_df = nuts3_df.rename(columns = {colname:colname.lower() for colname in nuts3_df.columns})
nuts3_df.set_index("nuts_id", drop=True, inplace=True)
nuts3_df

,cntr_code,nuts_name,population,area_km2,cropland_km2,cropland_area_percent,cropland_km2_by_type,name_latn,median_drought_days,max_drought_days,max_drought_days_year,median_warning_days,max_warning_days,max_warning_days_year,median_alert_days,max_alert_days,max_alert_days_year,geometry
nuts_id,,,,,,,,,,,,,,,,,,
AL011,AL,Dibër,104624,2470.31,23.12,0.94,"{'Maize': 10.12, 'Grapes': 0.13, 'Fruits': 8.5...",Dibër,55.32,232.66,2012.0,50.94,188.42,2012.0,5.24,44.24,2012.0,"POLYGON ((5180040.081 2144853.889, 5179764.448..."
AL012,AL,Durrës,222999,772.54,144.39,18.69,"{'Wheat': 19.74, 'Barley': 1.65, 'Maize': 28.9...",Durrës,141.75,294.40,2017.0,135.50,284.90,2017.0,4.62,48.46,2012.0,"POLYGON ((5139612.246 2104838.588, 5140583.791..."
AL013,AL,Kukës,60207,2391.45,8.19,0.34,"{'Other cereals': 0.4, 'Potatoes': 0.26, 'Rape...",Kukës,96.66,231.00,2017.0,85.48,205.54,2017.0,10.80,47.93,2012.0,"POLYGON ((5154970.036 2212802.267, 5156310.157..."
AL014,AL,Lezhë,96384,1662.35,111.27,6.69,"{'Wheat': 8.54, 'Barley': 1.55, 'Maize': 21.51...",Lezhë,150.61,268.43,2017.0,131.58,246.89,2017.0,13.59,41.72,2012.0,"POLYGON ((5166877.641 2159728.207, 5167535.432..."
AL015,AL,Shkodër,149496,3528.30,219.66,6.23,"{'Wheat': 5.85, 'Barley': 1.44, 'Maize': 35.03...",Shkodër,123.80,218.51,2017.0,103.34,197.15,2017.0,11.68,43.97,2012.0,"POLYGON ((5121233.536 2221719.441, 5120808.595..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
UKN0C,UK,Causeway Coast and Glens,NaN,1985.45,NaN,NaN,NaN,Causeway Coast and Glens,86.66,188.63,2018.0,82.46,183.29,2018.0,4.92,17.88,2013.0,"MULTIPOLYGON (((3286594.829 3688858.768, 32870..."
UKN0D,UK,Antrim and Newtownabbey,NaN,727.69,NaN,NaN,NaN,Antrim and Newtownabbey,96.77,248.28,2025.0,94.82,234.63,2025.0,6.00,17.95,2023.0,"POLYGON ((3291056.864 3638969.741, 3291998.505..."
UKN0E,UK,Lisburn and Castlereagh,NaN,510.17,NaN,NaN,NaN,Lisburn and Castlereagh,68.33,196.75,2025.0,65.85,192.63,2025.0,5.50,21.22,2022.0,"POLYGON ((3292499.936 3616588.715, 3292696.219..."


In [14]:
nuts3_df = nuts3_df.to_crs(epsg=4326)
nuts3_df.to_file(path.join(utils.out_data_dir, "nuts3_stats.geojson"))

In [16]:
groups = nuts3_df.groupby("cntr_code")
for ctr, df in groups:
    df.drop(columns=["geometry", "cntr_code"]).to_excel(path.join(utils.out_data_dir, "nuts3_stats_by_country", f"{ctr}.xlsx"))

In [17]:
nuts3_df

,cntr_code,nuts_name,population,area_km2,cropland_km2,cropland_area_percent,cropland_km2_by_type,name_latn,median_drought_days,max_drought_days,max_drought_days_year,median_warning_days,max_warning_days,max_warning_days_year,median_alert_days,max_alert_days,max_alert_days_year,geometry
nuts_id,,,,,,,,,,,,,,,,,,
AL011,AL,Dibër,104624,2470.31,23.12,0.94,"{'Maize': 10.12, 'Grapes': 0.13, 'Fruits': 8.5...",Dibër,55.32,232.66,2012.0,50.94,188.42,2012.0,5.24,44.24,2012.0,"POLYGON ((20.34672 41.87577, 20.34193 41.86729..."
AL012,AL,Durrës,222999,772.54,144.39,18.69,"{'Wheat': 19.74, 'Barley': 1.65, 'Maize': 28.9...",Durrës,141.75,294.40,2017.0,135.50,284.90,2017.0,4.62,48.46,2012.0,"POLYGON ((19.80665 41.56639, 19.81805 41.56442..."
AL013,AL,Kukës,60207,2391.45,8.19,0.34,"{'Other cereals': 0.4, 'Potatoes': 0.26, 'Rape...",Kukës,96.66,231.00,2017.0,85.48,205.54,2017.0,10.80,47.93,2012.0,"POLYGON ((20.14989 42.51392, 20.16495 42.50593..."
AL014,AL,Lezhë,96384,1662.35,111.27,6.69,"{'Wheat': 8.54, 'Barley': 1.55, 'Maize': 21.51...",Lezhë,150.61,268.43,2017.0,131.58,246.89,2017.0,13.59,41.72,2012.0,"POLYGON ((20.21222 42.02493, 20.21989 42.02303..."
AL015,AL,Shkodër,149496,3528.30,219.66,6.23,"{'Wheat': 5.85, 'Barley': 1.44, 'Maize': 35.03...",Shkodër,123.80,218.51,2017.0,103.34,197.15,2017.0,11.68,43.97,2012.0,"POLYGON ((19.75628 42.63384, 19.75005 42.62767..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
UKN0C,UK,Causeway Coast and Glens,NaN,1985.45,NaN,NaN,NaN,Causeway Coast and Glens,86.66,188.63,2018.0,82.46,183.29,2018.0,4.92,17.88,2013.0,"MULTIPOLYGON (((-6.41785 55.23746, -6.4091 55...."
UKN0D,UK,Antrim and Newtownabbey,NaN,727.69,NaN,NaN,NaN,Antrim and Newtownabbey,96.77,248.28,2025.0,94.82,234.63,2025.0,6.00,17.95,2023.0,"POLYGON ((-6.1708 54.81143, -6.15513 54.80998,..."
UKN0E,UK,Lisburn and Castlereagh,NaN,510.17,NaN,NaN,NaN,Lisburn and Castlereagh,68.33,196.75,2025.0,65.85,192.63,2025.0,5.50,21.22,2022.0,"POLYGON ((-6.07034 54.61901, -6.06527 54.61414..."


In [18]:
lau_lookup_table = {}


for ctr in list(set((nuts3_df["cntr_code"].values))):
    if ctr == "UK":
        pass
    else:
        lookup_df = pd.read_excel(path.join(utils.raw_data_dir, "EU-27-LAU-2023-NUTS-2021.xlsx"), sheet_name=ctr)
        break

In [ ]:
eu_nats_lau_lookup

NameError: name 'eu_nats_lau_lookup' is not defined